In [ ]:
# print("hello")

In [ ]:
# import idx2numpy
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Loading the training images and labels

file_name = "mnist.npz"

with np.load(file_name) as data:
    X_train = data['x_train']
    y_train = data['y_train']
    X_test = data['x_test']
    y_test = data['y_test']


print(X_train.shape)
print(X_test.shape)

In [ ]:
filter=np.isin(y_train,[0,1,2])
test_filter = np.isin(y_test,[0,1,2])



X_train_sampled = X_train[filter]
y_train_final = y_train[filter]


X_test_sampled = X_test[test_filter]
y_test_final = y_test[test_filter]

print(X_train_sampled.shape)
print(X_test_sampled.shape)

In [ ]:

no_train_observations=X_train_sampled.shape[0]
no_test_observations=X_test_sampled.shape[0]

# REPLACE the converrt function and its usage with this:
X_train_final = (X_train_sampled.reshape(no_train_observations, -1) / 255.0).T  #shape (784, N)
X_test_final = (X_test_sampled.reshape(no_test_observations, -1) / 255.0).T

print(X_train_final.shape)
print(X_test_final.shape)

## Finding cov and Mean

In [ ]:
mean=np.zeros((784,1))

for i in range(784):
    sum=0
    for j in range(no_train_observations):
        sum+=X_train_final[i][j]
    mean[i][0]=sum/no_train_observations    

X_c=X_train_final-mean

sigma=(X_c @ X_c.T)/(no_train_observations-1)

identity=np.eye(784)

print(sigma.shape)

## Finding eigen val and vectors

In [ ]:
eigenvalues,eigenvectors=np.linalg.eigh(sigma)

print(eigenvalues.shape)
print(eigenvectors.shape)

eig=[]

for i in range(784):
    eig.append((eigenvalues[i],eigenvectors[:,i]))

eig.sort(key=lambda x: x[0], reverse=True)


sor_values=np.sort(eigenvalues)[::-1]

tot_sum=np.sum(eigenvalues)
# ret= tot_sum*0.75  # Retaining 75% variance

p=10
# curr_sum=0
# while(p<784 and curr_sum<ret):
#     curr_sum+=sor_values[p]
#     p+=1

print("No. of components : ",p)


U=eig[0][1].reshape(-1, 1)


for i in range(1,p):
    new_col = eig[i][1].reshape(-1, 1)
    U = np.hstack((U, new_col))

print(U.shape)
    

## Obtaining the projected centered matrix

In [ ]:
Y = U.T @ X_c
print(Y.shape)

## Defining Target Vectors

In [ ]:
y_0 = (y_train_final == 0).astype(int) 
y_1 = (y_train_final == 1).astype(int)
y_2 = (y_train_final == 2).astype(int)

In [ ]:
# Transpose to get shape (18623, 10)
X_train_pca = Y.T 

# Center and project the test set using the same mean and eigenvectors (U)
X_test_c = X_test_final - mean
X_test_pca = (U.T @ X_test_c).T

print("PCA Train shape:", X_train_pca.shape)
print("PCA Test shape:", X_test_pca.shape)


y_train_tree = y_train_final
y_test_tree = y_test_final

## Calulation

In [ ]:
#Calculates the Gini impurity for a given node
def calculate_gini(y):    
    if len(y) == 0:
        return 0.0
    # Find the proportion of each class in the node
    _, counts=np.unique(y, return_counts=True)
    probabilities=counts/len(y)
    gini=1.0 - np.sum(probabilities ** 2)
    return gini

#calculates the weighted Gini impurity after a split
def calculate_weighted_gini(y_left, y_right):
    n_total=len(y_left) + len(y_right)
    if n_total == 0:
        return 0.0
    
    p_left=len(y_left)/n_total
    p_right=len(y_right)/n_total
    
    weighted_gini=p_left * calculate_gini(y_left) + p_right * calculate_gini(y_right)
    return weighted_gini

### Finding Best Split

In [ ]:
# Finds optimal split

def get_best_split(X, y, n_thresholds=18000):
    best_gini=float('inf')
    best_feature=None
    best_threshold=None

    for feature_idx in range(X.shape[1]):
        feature_values=X[:, feature_idx]
        sorted_vals=np.unique(feature_values)
        midpoints=(sorted_vals[:-1] + sorted_vals[1:]) / 2.0
        if len(midpoints) > n_thresholds:
            idx=np.linspace(0, len(midpoints)-1, n_thresholds, dtype=int)
            thresholds=midpoints[idx]
        else:
            thresholds=midpoints

        for threshold in thresholds:
            left_mask=feature_values <= threshold
            right_mask=~left_mask
            y_left, y_right=y[left_mask], y[right_mask]
            if len(y_left) == 0 or len(y_right) == 0:
                continue
            gini=calculate_weighted_gini(y_left, y_right)
            if gini < best_gini:
                best_gini=gini
                best_feature=feature_idx
                best_threshold=threshold

    return best_feature, best_threshold, best_gini


In [ ]:

best_f_reported, best_t_reported, best_gini_reported = get_best_split(X_train_pca, y_train_tree)

f1 = best_f_reported
t1 = best_t_reported 
left_mask_tmp = X_train_pca[:, f1] <= t1
right_mask_tmp = X_train_pca[:, f1] > t1
g1 = calculate_weighted_gini(y_train_tree[left_mask_tmp], y_train_tree[right_mask_tmp])
print(f"First Split: Feature {f1} at threshold {t1:.4f} (Weighted Gini: {g1:.4f})")

left_mask_1 = X_train_pca[:, f1] <= t1
right_mask_1 = X_train_pca[:, f1] > t1

X_left_1, y_left_1 = X_train_pca[left_mask_1], y_train_tree[left_mask_1]
X_right_1, y_right_1 = X_train_pca[right_mask_1], y_train_tree[right_mask_1]

#second split
#we need to decide which region to split next

gini_left_1 = calculate_gini(y_left_1)
gini_right_1 = calculate_gini(y_right_1)

if gini_left_1 > gini_right_1:
    print("Splitting the Left Region")
    f2, t2, g2 = get_best_split(X_left_1, y_left_1)
    split_node = 'left'
else:
    print("Splitting the Right Region")
    f2, t2, g2 = get_best_split(X_right_1, y_right_1)
    split_node = 'right'

print(f"Second Split: Feature {f2} at threshold {t2:.4f} (Weighted Gini: {g2:.4f})")

#determine majority classes for the 3 terminal nodes
def get_majority_class(y):
    values, counts = np.unique(y, return_counts=True)
    return values[np.argmax(counts)]

if split_node == 'left':
    left_mask_2 = X_left_1[:, f2] <= t2
    right_mask_2 = X_left_1[:, f2] > t2
    
    leaf_A_class = get_majority_class(y_left_1[left_mask_2])
    leaf_B_class = get_majority_class(y_left_1[right_mask_2])
    leaf_C_class = get_majority_class(y_right_1)
else:
    left_mask_2 = X_right_1[:, f2] <= t2
    right_mask_2 = X_right_1[:, f2] > t2
    
    leaf_A_class = get_majority_class(y_left_1)
    leaf_B_class = get_majority_class(y_right_1[left_mask_2])
    leaf_C_class = get_majority_class(y_right_1[right_mask_2])


print(f"Terminal Node Majority Classes: {leaf_A_class}, {leaf_B_class}, {leaf_C_class}")

In [ ]:

#traverses the 3-node tree for a single sample
def predict_tree(x, f1, t1, f2, t2, split_node, leaf_A_class, leaf_B_class, leaf_C_class):
    if split_node == 'left':
        if x[f1] <= t1:
            return leaf_A_class if x[f2] <= t2 else leaf_B_class
        else:
            return leaf_C_class
    else:
        if x[f1] <= t1:
            return leaf_A_class
        else:
            return leaf_B_class if x[f2] <= t2 else leaf_C_class

#predict on the test set
predictions_list=[]
for x in X_test_pca:
    prediction=predict_tree(x,f1, t1,f2, t2,split_node,leaf_A_class,leaf_B_class,leaf_C_class)

    predictions_list.append(prediction)

#convert the final list back into a NumPy array
y_pred=np.array(predictions_list)

#overall Accuracy
overall_acc=np.mean(y_pred == y_test_tree) * 100
print(f"Overall Test Accuracy: {overall_acc:.2f}%")

#class wise Accuracy
for cls in [0, 1, 2]:
    cls_mask=(y_test_tree == cls)
    cls_acc=np.mean(y_pred[cls_mask] == y_test_tree[cls_mask]) * 100
    print(f"Accuracy for Class {cls}: {cls_acc:.2f}%")

In [ ]:

#trains a 3 terminal node decision tree and returns its structure
def train_3_node_tree(X, y):
    f1, t1, _=get_best_split(X, y)
    
    left_mask_1=X[:, f1] <= t1
    right_mask_1=X[:, f1] > t1
    X_left_1, y_left_1=X[left_mask_1], y[left_mask_1]
    X_right_1, y_right_1=X[right_mask_1], y[right_mask_1]

    gini_left_1=calculate_gini(y_left_1)
    gini_right_1=calculate_gini(y_right_1)

    if gini_left_1 > gini_right_1:
        f2, t2, _=get_best_split(X_left_1, y_left_1)
        split_node='left'
        
        left_mask_2=X_left_1[:, f2] <= t2
        right_mask_2=X_left_1[:, f2] > t2
        leaf_A=get_majority_class(y_left_1[left_mask_2])
        leaf_B=get_majority_class(y_left_1[right_mask_2])
        leaf_C=get_majority_class(y_right_1)
    else:
        f2, t2, _=get_best_split(X_right_1, y_right_1)
        split_node='right'
        
        left_mask_2=X_right_1[:, f2] <= t2
        right_mask_2=X_right_1[:, f2] > t2
        leaf_A=get_majority_class(y_left_1)
        leaf_B=get_majority_class(y_right_1[left_mask_2])
        leaf_C=get_majority_class(y_right_1[right_mask_2])

    return {
        'f1': f1, 't1': t1, 'f2': f2, 't2': t2,
        'split_node': split_node,
        'leaf_A': leaf_A, 'leaf_B': leaf_B, 'leaf_C': leaf_C
    }

#Predicts the class for a single sample using a trained tree
def predict_single_tree(tree, x):
    if tree['split_node'] == 'left':
        if x[tree['f1']] <= tree['t1']:
            return tree['leaf_A'] if x[tree['f2']] <= tree['t2'] else tree['leaf_B']
        else:
            return tree['leaf_C']
    else:
        if x[tree['f1']] <= tree['t1']:
            return tree['leaf_A']
        else:
            return tree['leaf_B'] if x[tree['f2']] <= tree['t2'] else tree['leaf_C']

## BAGGING and oob

In [ ]:
np.random.seed(42)
n_samples=X_train_pca.shape[0]
n_trees=5
bagged_trees=[]
oob_errors=[]

all_boot_indices = []

for i in range(n_trees):
    #Generate a random set of indices with replacement
    #n_samples is the total size dataset
    boot_indices = np.random.choice(n_samples, size=n_samples, replace=True)
    
    #Store this specific set of indices in our list
    all_boot_indices.append(boot_indices)

print("Training Bagged Trees")
for i in range(n_trees):
    #create Bootstrap Dataset (sampling with replacement)
    boot_indices=all_boot_indices[i]
    
    #find OOB indices
    # np.setdiff1d finds unique elements in array1 that are not in array2
    oob_indices=np.setdiff1d(np.arange(n_samples),boot_indices)
    
    X_boot,y_boot=X_train_pca[boot_indices],y_train_tree[boot_indices]
    X_oob,y_oob=X_train_pca[oob_indices],y_train_tree[oob_indices]
    
    #train the tree on the bootstrap dataset
    tree=train_3_node_tree(X_boot,y_boot)
    bagged_trees.append(tree)
    
    #calculate OOB error for this tree
    y_oob_pred=np.array([predict_single_tree(tree,x) for x in X_oob])
    oob_error=np.mean(y_oob_pred != y_oob)
    oob_errors.append(oob_error)
    
    print(f"Tree {i+1} trained. OOB Error: {oob_error:.4f}")

#report Average OOB Error
average_oob_error=np.mean(oob_errors)
print(f"\nAverage OOB Error across 5 trees: {average_oob_error:.4f}")

report

In [ ]:
print("\nEvaluating Bagged Ensemble on Test Set")

#get predictions from all 5 trees for every test sample
#shape will be (5,num_test_samples)
all_tree_predictions_list=[]

for tree in bagged_trees:
    current_tree_results=[]
    for x in X_test_pca:
        #get the prediction from the specific tree for the specific sample
        prediction=predict_single_tree(tree, x)
        current_tree_results.append(prediction)
    
    #add the full set of test predictions for this tree to the main list
    all_tree_predictions_list.append(current_tree_results)

all_tree_predictions=np.array(all_tree_predictions_list)

#for each column (test sample), count occurrences of 0,1,2
#then find the index of the max count using argmax
final_bagged_preds=np.array([np.argmax(np.bincount(all_tree_predictions[:, i])) for i in range(all_tree_predictions.shape[1])])

#calculate accuracies
bagged_overall_acc=np.mean(final_bagged_preds == y_test_tree)*100
print(f"Bagging Overall Test Accuracy: {bagged_overall_acc:.2f}%")

for cls in [0, 1, 2]:
    cls_mask=(y_test_tree == cls)
    cls_acc=np.mean(final_bagged_preds[cls_mask] == y_test_tree[cls_mask])*100
    print(f"Bagging Accuracy for Class {cls}: {cls_acc:.2f}%")

## Random forest

In [ ]:
def get_best_split_rf(X,y,k_features,n_thresholds=18000):
    best_gini=float('inf')
    best_feature=None
    best_threshold=None

    n_features=X.shape[1]
    feature_indices=np.random.choice(n_features,k_features,replace=False)

    for feature_idx in feature_indices:
        feature_values=X[:,feature_idx]
        sorted_vals=np.unique(feature_values)
        midpnt=(sorted_vals[:-1] + sorted_vals[1:])/2.0
        if len(midpnt)>n_thresholds:
            idx=np.linspace(0,len(midpnt)-1,n_thresholds,dtype=int)
            thresholds=midpnt[idx]
        else:
            thresholds=midpnt

        for threshold in thresholds:
            left_mask=feature_values <= threshold
            right_mask=~left_mask
            y_left=y[left_mask]
            y_right=y[right_mask]
            if len(y_left) == 0 or len(y_right) == 0:
                continue
            gini=calculate_weighted_gini(y_left,y_right)
            if gini < best_gini:
                best_gini=gini
                best_feature=feature_idx
                best_threshold=threshold

    return best_feature,best_threshold,best_gini


#trains a 3 node tree using Random Forest feature selection
def train_rf_tree(X,y,k_features):
  
    f1,t1,_=get_best_split_rf(X,y,k_features)

    left_mask_1=X[:,f1] <= t1
    right_mask_1=X[:,f1]>t1
    X_left_1,y_left_1=X[left_mask_1],y[left_mask_1]
    X_right_1,y_right_1=X[right_mask_1],y[right_mask_1]

    gini_left_1=calculate_gini(y_left_1)
    gini_right_1=calculate_gini(y_right_1)

    if gini_left_1>gini_right_1:
        f2,t2,_=get_best_split_rf(X_left_1,y_left_1,k_features)
        split_node='left'
        left_mask_2=X_left_1[:,f2] <= t2
        right_mask_2=X_left_1[:,f2]>t2
        leaf_A=get_majority_class(y_left_1[left_mask_2])
        leaf_B=get_majority_class(y_left_1[right_mask_2])
        leaf_C=get_majority_class(y_right_1)
    else:
        f2,t2,_=get_best_split_rf(X_right_1,y_right_1,k_features)
        split_node='right'
        left_mask_2=X_right_1[:,f2] <= t2
        right_mask_2=X_right_1[:,f2]>t2
        leaf_A=get_majority_class(y_left_1)
        leaf_B=get_majority_class(y_right_1[left_mask_2])
        leaf_C=get_majority_class(y_right_1[right_mask_2])

    return {'f1': f1,'t1': t1,'f2': f2,'t2': t2,'split_node': split_node,'leaf_A': leaf_A,'leaf_B': leaf_B,'leaf_C': leaf_C}


In [ ]:

k_features=3 #(sqrt of 10)
rf_trees=[]
rf_oob_errors=[]

print(f"Training Random Forest (k={k_features})...")
for i in range(n_trees):
    boot_indices=boot_indices=all_boot_indices[i]
    oob_indices=np.setdiff1d(np.arange(n_samples),boot_indices)
    
    X_boot,y_boot=X_train_pca[boot_indices],y_train_tree[boot_indices]
    X_oob,y_oob=X_train_pca[oob_indices],y_train_tree[oob_indices]
    
    # Train using the RF function
    tree=train_rf_tree(X_boot,y_boot,k_features)
    rf_trees.append(tree)
    
    #calculate OOB error
    y_oob_pred=np.array([predict_single_tree(tree,x) for x in X_oob])
    oob_error=np.mean(y_oob_pred != y_oob)
    rf_oob_errors.append(oob_error)
    
    print(f"RF Tree {i+1} trained. OOB Error: {oob_error:.4f}")

#average OOB Error
print(f"\nAverage RF OOB Error across 5 trees: {np.mean(rf_oob_errors):.4f}")

#ensemble evaluation
print("\nEvaluating Random Forest on Test Set")


all_rf_predictions_list=[]

for tree in rf_trees:
    lis=[]
    for x in X_test_pca:
        vote=predict_single_tree(tree,x)
        lis.append(vote)
    
    all_rf_predictions_list.append(lis)

all_rf_predictions=np.array(all_rf_predictions_list)

num_samples = all_rf_predictions.shape[1]

final_rf_preds_list = []

for i in range(num_samples):
    sample_predictions = all_rf_predictions[:, i]
    

    counts = np.bincount(sample_predictions)
    winner = np.argmax(counts)
    final_rf_preds_list.append(winner)

final_rf_preds = np.array(final_rf_preds_list)




rf_overall_acc=np.mean(final_rf_preds == y_test_tree)*100
print(f"Random Forest Overall Test Accuracy: {rf_overall_acc:.2f}%")

for cls in [0,1,2]:
    cls_mask=(y_test_tree == cls)
    cls_acc=np.mean(final_rf_preds[cls_mask] == y_test_tree[cls_mask])*100
    print(f"Random Forest Accuracy for Class {cls}: {cls_acc:.2f}%")


In [ ]:
print("\n COMPARISON: Bagging vs Random Forest ")
print(f"Bagging,Test Acc: {bagged_overall_acc:.2f}%,Avg OOB: {average_oob_error:.4f}")
print(f"RF (k=3),Test Acc: {rf_overall_acc:.2f}%,Avg OOB: {np.mean(rf_oob_errors):.4f}")